# Distributed Shared Storage — Quick Start

The simplest way to get distributed shared storage on your FABRIC nodes.

Just pass **`storage=True`** when creating your slice — FABlib handles everything else
(networking, credentials, mounting).

### Prerequisites
- Your project must have storage allocated. To request storage, go to **Portal → Experiments → Projects → *Your Project* → Request Storage**.
- Complete the [Configure Environment](../../../configure_and_validate.ipynb) notebook first

> For advanced demos (multi-site sharing, persistence across slices, data pipelines),
> see [cephfs_storage.ipynb](./cephfs_storage.ipynb).

## 1. Setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config();

## 2a. Slice-Level Storage

Pass `storage=True` to `new_slice()` to enable shared storage on **every** node in the slice.

In [ ]:
SLICE_NAME = "CephFS-QuickStart"
SITE = "STAR"

slice1 = fablib.new_slice(name=SLICE_NAME, storage=True)
node1 = slice1.add_node(name="node1", site=SITE, cores=4, ram=8, disk=50)
slice1.submit();

### Verify the Storage Mount

In [ ]:
slice1 = fablib.get_slice(name=SLICE_NAME)
node1 = slice1.get_node("node1")

print(f"Storage enabled: {node1.has_storage()}")
print(f"Cluster: {node1.get_storage_cluster()}")

node1.execute("df -h | grep ceph")

### Use Shared Storage

Read and write files on the shared storage mount just like any local directory.

In [ ]:
# Discover the mount path
stdout, _ = node1.execute("mount | grep ceph | awk '{print $3}' | head -1", quiet=True)
mount_path = stdout.strip()
print(f"CephFS mount path: {mount_path}")

# Write a file
node1.execute(f"echo 'Hello from FABRIC CephFS!' > {mount_path}/hello.txt")

# Read it back
node1.execute(f"cat {mount_path}/hello.txt")

# List contents
node1.execute(f"ls -lh {mount_path}/")

## 3a. Clean Up Slice-Level Example

In [ ]:
node1.execute(f"rm -f {mount_path}/hello.txt", quiet=True)
slice1.delete()
print("Slice deleted.")

---
## 2b. Node-Level Storage

Use `storage=True` on individual `add_node()` calls to enable shared storage only on specific nodes.
This is useful when only some nodes need shared storage.

In [ ]:
SLICE_NAME_2 = "CephFS-NodeLevel"

slice2 = fablib.new_slice(name=SLICE_NAME_2)

# Only this node gets shared storage
storage_node = slice2.add_node(name="with-storage", site="STAR", cores=4, ram=8, disk=50, storage=True)

# This node does NOT get shared storage
compute_node = slice2.add_node(name="compute-only", site="STAR", cores=4, ram=8, disk=50)

slice2.submit();

### Verify selective storage

In [ ]:
slice2 = fablib.get_slice(name=SLICE_NAME_2)

for node in slice2.get_nodes():
    has = node.has_storage()
    cluster = node.get_storage_cluster() if has else "N/A"
    print(f"{node.get_name():20s}  storage={has!s:5s}  cluster={cluster}")
    node.execute("df -h | grep ceph || echo '  No CephFS mount'")

### Clean Up Node-Level Example

In [ ]:
slice2.delete()
print("Slice deleted.")

## Quick Reference

| What | How |
|------|-----|
| Enable for all nodes | `fablib.new_slice(name="...", storage=True)` |
| Enable per node | `slice.add_node(name="...", storage=True)` |
| Pick a cluster | `fablib.new_slice(storage=True, storage_cluster="east")` |
| Check if enabled | `node.has_storage()` |
| Get cluster name | `node.get_storage_cluster()` |

### Next steps

- [cephfs_storage.ipynb](./cephfs_storage.ipynb) — Multi-site sharing, persistence across slices, data pipelines
- [Distributed Storage Benchmarking](../../complex_recipes/cephfs_benchmarking/) — Throughput benchmarks and performance tuning